# Abalone Age Prediction System – 10-Fold Cross Validation (Full Dataset)

## Setup: Running This Notebook on Google Colab

1. Go to https://colab.research.google.com and upload this `.ipynb` notebook.
2. Run the notebook cells in order using **Runtime → Run all**.
3. When Step 2.1 asks for the dataset, upload **`abalone.csv`**.
4. The notebook compares five regression algorithms using 10-fold cross-validation, trains the best one on the full dataset, and predicts age in years for a new abalone.

The `abalone.csv` file is included separately with the notebook.


## Introduction

##### **Definition**
This is a Machine Learning regression task that estimates an abalone's age in years from its physical measurements: sex, length, diameter, height, whole weight, shucked weight, viscera weight, and shell weight.

##### **Dataset target**
The dataset provides a `Rings` count. This notebook converts it to an estimated age using **Age (years) = Rings + 1.5**, then trains the model to predict that age.

### Aim of This Notebook

- Load and prepare the Abalone dataset.
- Compare five regression algorithms using 10-fold cross-validation.
- Select the algorithm with the lowest average RMSE.
- Train the selected model on the full dataset and save it.
- Accept measurements for a new abalone and display its **predicted age in years**.

### Expected output
For a new abalone, the application displays a prediction such as `Predicted Abalone Age: 9.50 years`. This is an illustrative format; the actual prediction depends on the input and trained model.


# Code – Abalone Age Prediction System
## Step 1: Import Libraries

In [ ]:
# --- Core Python Libraries ---
import os
import pickle
import time

import numpy as np
import pandas as pd

# --- Machine Learning (scikit-learn) ---
import sklearn
from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# --- The 5 Regression Algorithms ---
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# --- Visualization ---
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator

# --- Progress bar (shows elapsed and remaining time) ---
from tqdm import tqdm

# Show large numbers with thousands separators in pandas tables
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

print("Library versions used in this run:")
print(f"  NumPy        : {np.__version__}")
print(f"  Pandas       : {pd.__version__}")
print(f"  scikit-learn : {sklearn.__version__}")
print(f"  Matplotlib   : {matplotlib.__version__}")

# Step 2: Load Sample Data
## Step 2.1: Upload the Dataset (Google Colab)

When prompted, select **`abalone.csv`**. If the file is already present in the working directory, upload is skipped.


In [ ]:
DATA_FILE = "abalone.csv"

# google.colab is only available inside Google Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not os.path.exists(DATA_FILE):
    print(f"Please click 'Choose Files' and select '{DATA_FILE}' ...")
    uploaded = files.upload()
    uploaded_name = list(uploaded.keys())[0]
    if uploaded_name != DATA_FILE:
        os.rename(uploaded_name, DATA_FILE)

if os.path.exists(DATA_FILE):
    print(f"'{DATA_FILE}' is ready to use.")
else:
    raise FileNotFoundError(f"'{DATA_FILE}' not found. Please upload the dataset and run this cell again.")


## Step 2.2: Read the Dataset

This notebook uses the full Abalone dataset. The target is created from the provided ring count using the common estimate `Age (years) = Rings + 1.5`.


In [ ]:
# Load the full Abalone dataset
abalone = pd.read_csv(DATA_FILE)

# The original dataset stores ring count. Estimate age in years using Rings + 1.5.
abalone["Age (years)"] = abalone["Rings"] + 1.5

print(f"Loaded dataset: {abalone.shape[0]:,} abalones")

# Preview the dataset
abalone.head()


# Step 3: Understand and Pre-process Sample Data
## Step 3.1: Understand Sample Data

In [ ]:
"""
Purpose of this section:
-------------------------
This part of the code helps us understand the dataset by:
1. Printing the names of all attributes (columns).
2. Showing the total number of instances (rows).
3. Showing how many values are missing in each attribute.
"""

print("\nAttributes in Sample Data:")
print("==========================\n")
print(list(abalone.columns))

print("\nNumber of Instances in Sample Data:")
print("===================================\n")
print(len(abalone))

print("\nMissing Values in Each Attribute:")
print("=================================\n")
missing = abalone.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values")

### Understanding the Output (Age in Years)

The dataset's original target is the number of shell rings. Here, the target is converted to estimated age in years by adding 1.5 to the ring count. The model therefore learns to predict age in years rather than the raw ring count.


In [ ]:
# Summary statistics of the estimated abalone age (years)
age_summary = abalone["Age (years)"].describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99])
age_summary.to_frame(name="Age (years)")


> **Why RMSE matters:** Root Mean Squared Error (RMSE) measures the typical size of prediction errors, in years. Larger errors are penalized more strongly because the errors are squared before averaging. Lower RMSE indicates better predictions.


## Step 3.2: Pre-process Sample Data

In [ ]:
"""
Pre-processing steps:
---------------------
1. Remove duplicate rows.
2. Missing numeric values are filled inside the pipeline using the training fold's median.
3. Missing categorical values are filled inside the pipeline using the most frequent category.
"""

rows_before = len(abalone)
abalone = abalone.drop_duplicates().reset_index(drop=True)
print(f"Duplicate rows removed: {rows_before - len(abalone):,}")
print(f"Instances remaining   : {len(abalone):,}")

print("\nMissing values will be handled by the preprocessing pipeline.")


# Step 4: Feature Extraction

Use the seven numeric measurements and the categorical `Sex` field as input features. The output target is estimated **Age (years)**.


In [ ]:
numeric_features = [
    "Length", "Diameter", "Height", "Whole weight",
    "Shucked weight", "Viscera weight", "Shell weight"
]
categorical_features = ["Sex"]
target = "Age (years)"

# Input features (X) and output (y)
X = abalone[numeric_features + categorical_features]
y = abalone[target]

print(f"Input features (X): {X.shape[1]} columns -> {list(X.columns)}")
print(f"Output (y)        : {target}")
print(f"Number of abalones: {len(X):,}")

X.head()


# Step 5: Build the Pre-processing Pipeline

Machine learning algorithms need numbers, and some algorithms (KNN, Linear Regression) are sensitive to the scale of each feature. The pipeline does three things:

| Feature type | Step | Why |
|---|---|---|
| Numeric | **Fill missing values** with the median | Algorithms cannot handle empty values |
| Numeric | **Standard scaling** (mean 0, std 1) | So that large-scale features such as *Whole weight* do not dominate small-scale ones such as *Height* |
| Categorical | **One-hot encoding** | Turns text categories into 0/1 columns. Unlike label encoding, it does not invent a fake order among the *Sex* categories (M, F, I) |

### Why put pre-processing *inside* the pipeline?

In the previous notebook, the encoders were fitted on the **whole** dataset before splitting. With cross validation this would cause **data leakage**: the scaler and imputer would "see" the test fold.

By placing them inside the pipeline, the cross validation loop in Step 8 **re-fits them on the training folds only** in every round, and then applies them to the test fold. This is exactly what happens in the real world, where a model never sees future data during training.

`handle_unknown="ignore"` means that if a test fold contains a category that never appeared in the training folds, the model still works and simply treats it as "no known category".

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

print("Pre-processing pipeline created:")
preprocessor

# Step 6: Define the 5 Regression Algorithms

We compare five regression algorithms:

1. **Linear Regression** – models a weighted linear relationship between measurements and age.
2. **K-Nearest Neighbors (KNN)** – predicts age using the ages of similar abalones.
3. **Decision Tree** – learns measurement-based decision rules.
4. **Random Forest** – combines predictions from many decision trees.
5. **Gradient Boosting** – builds trees sequentially to improve earlier predictions.

The algorithm with the lowest mean cross-validation RMSE is selected.


In [ ]:
algorithms = {
    "Linear Regression": LinearRegression(),
    "K-Nearest Neighbors": KNeighborsRegressor(n_neighbors=5),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

print("Algorithms to compare:")
for number, name in enumerate(algorithms, start=1):
    print(f"  {number}. {name}")

# Step 7: Set Up 10-Fold Cross Validation
## Step 7.1: Create the K-Fold Splitter

- `n_splits=10`: split the dataset into 10 folds.
- `shuffle=True`: shuffle samples before splitting.
- `random_state=42`: make the split reproducible.

The same folds are used for all five algorithms so their results can be compared fairly.


In [ ]:
K = 10
kfold = KFold(n_splits=K, shuffle=True, random_state=42)

print(f"{K}-Fold Cross Validation created (shuffle=True, random_state=42)")

## Step 7.2: Inspect the Folds

Let's check how many abalone are used for training and testing in each round.

In [ ]:
fold_sizes = []
for fold_number, (train_index, test_index) in enumerate(kfold.split(X), start=1):
    fold_sizes.append({
        "Fold (Round)": fold_number,
        "Training samples": len(train_index),
        "Testing samples": len(test_index),
        "Average test age (years)": y.iloc[test_index].mean(),
        "Maximum test age (years)": y.iloc[test_index].max(),
    })

fold_sizes_table = pd.DataFrame(fold_sizes).set_index("Fold (Round)")
fold_sizes_table


The average and maximum estimated ages differ across test folds. A fold with more difficult examples may have a higher RMSE.

# Step 8: Run 10-Fold Cross Validation for All 5 Algorithms

Each algorithm is trained on nine folds and evaluated on the remaining fold. This repeats ten times, with a fresh pipeline each time. RMSE is measured in years.


In [ ]:
fold_rmse = {}        # RMSE on each fold, for each algorithm
training_time = {}    # total time taken for all folds

folds = list(kfold.split(X))     # the same folds for every algorithm
total_steps = len(algorithms) * K

with tqdm(total=total_steps, unit="fold") as progress_bar:
    for name, algorithm in algorithms.items():
        model = Pipeline([
            ("preprocessor", preprocessor),
            ("regressor", algorithm),
        ])

        scores = []
        start = time.time()
        for fold_number, (train_index, test_index) in enumerate(folds, start=1):
            progress_bar.set_description(f"{name}, fold {fold_number}/{K}")
            fold_model = clone(model)
            fold_model.fit(X.iloc[train_index], y.iloc[train_index])
            predictions = fold_model.predict(X.iloc[test_index])
            rmse = np.sqrt(mean_squared_error(y.iloc[test_index], predictions))
            scores.append(rmse)
            progress_bar.update(1)

        training_time[name] = time.time() - start
        fold_rmse[name] = np.array(scores)
        tqdm.write(f"{name:<20} -> mean RMSE = {fold_rmse[name].mean():.3f} years   ({training_time[name]:.1f} s)")

    progress_bar.set_description("Done")

print(f"\n{K}-fold cross validation completed for all {len(algorithms)} algorithms.")


# Step 9: Compare the Algorithms
## Step 9.1: RMSE of Each Algorithm on Each Fold

Each row is one round of cross validation, and each column is one algorithm. The **lowest RMSE in each row** (the best algorithm on that fold) is highlighted in green.

**How to read it:** count the green cells in each column to see how many folds each algorithm **won**. The algorithm with the best *average* does not always win every fold, which is why we compare the mean over all folds (Step 9.2) rather than trusting any single fold.

In [ ]:
per_fold_table = pd.DataFrame(fold_rmse)
per_fold_table.index = [f"Fold {i}" for i in range(1, K + 1)]
per_fold_table.index.name = "Fold"

(per_fold_table.style
    .format("{:.3f}")
    .highlight_min(axis=1, props="background-color: #c8ebd2; color: #0b0b0b; font-weight: bold;"))


## Step 9.2: Summary Comparison Table

This is the main result of the notebook. For each algorithm:

| Column | Meaning |
|---|---|
| **Mean RMSE** | Average RMSE over the 10 folds. This is the **main measure for ranking** the algorithms (lower is better) |
| **Std RMSE** | Standard deviation over the folds. A **small value means the algorithm is consistent**; a large value means its performance depends on which abalone it gets |
| **Best / Worst Fold** | Lowest and highest RMSE across the folds |
| **RMSE as % of Avg Age** | Mean RMSE divided by the average abalone age. This puts the error in context |
| **Time** | Total time to train and test across all 10 folds |
| **Rank** | 1 = lowest mean RMSE = best algorithm |

In [ ]:
average_age = y.mean()

summary = pd.DataFrame({
    "Mean RMSE (years)": per_fold_table.mean(),
    "Std RMSE (years)": per_fold_table.std(),
    "Best Fold RMSE (years)": per_fold_table.min(),
    "Worst Fold RMSE (years)": per_fold_table.max(),
    "RMSE as % of Avg Age": per_fold_table.mean() / average_age * 100,
    "Time (s)": pd.Series(training_time),
})
summary = summary.sort_values("Mean RMSE (years)")
summary.insert(0, "Rank", range(1, len(summary) + 1))
summary.index.name = "Algorithm"

print(f"Average estimated abalone age: {average_age:.2f} years\n")

error_columns = ["Mean RMSE (years)", "Std RMSE (years)", "Best Fold RMSE (years)", "Worst Fold RMSE (years)"]
(summary.style
    .format({**{c: "{:.3f}" for c in error_columns},
             "RMSE as % of Avg Age": "{:.1f}%",
             "Time (s)": "{:.2f}"})
    .highlight_min(subset=["Mean RMSE (years)"], props="background-color: #c8ebd2; color: #0b0b0b; font-weight: bold;"))


## Step 9.3: Visual Comparison

**Chart 1** shows the mean RMSE of each algorithm (shorter bar = better). The black whisker shows ± 1 standard deviation across the 10 folds.

**Chart 2** shows the RMSE of **every individual fold** as a dot, so you can see how spread out each algorithm's results are.

In [ ]:
# Chart styling
BAR_COLOR = "#2a78d6"
TEXT_COLOR = "#0b0b0b"
MUTED_TEXT = "#52514e"
GRID_COLOR = "#e4e3df"

def style_axis(ax):
    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID_COLOR)
    ax.tick_params(colors=MUTED_TEXT, length=0)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.grid(axis="x", color=GRID_COLOR, linewidth=0.8)
    ax.set_axisbelow(True)

order = summary.index.tolist()
positions = np.arange(len(order))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.2))
means = summary["Mean RMSE (years)"]
stds = summary["Std RMSE (years)"]
ax1.barh(positions, means, height=0.5, color=BAR_COLOR,
         xerr=stds, error_kw=dict(ecolor=TEXT_COLOR, elinewidth=1.2, capsize=4))
for i, name in enumerate(order):
    label = f"{means[name]:.2f}" + ("   (best)" if i == 0 else "")
    ax1.text(means[name] + stds[name] + means.max() * 0.02, i, label,
             va="center", color=TEXT_COLOR, fontsize=10)
ax1.set_yticks(positions, order, color=TEXT_COLOR)
ax1.invert_yaxis()
ax1.set_xlim(0, (means + stds).max() * 1.3)
style_axis(ax1)
ax1.set_title("Mean RMSE across 10 folds (lower is better)", loc="left", color=TEXT_COLOR, fontsize=12)
ax1.set_xlabel("RMSE (years)", color=MUTED_TEXT)

for i, name in enumerate(order):
    ax2.scatter(per_fold_table[name], np.full(K, i), s=60, color=BAR_COLOR,
                edgecolor="white", linewidth=1.5, zorder=3,
                label="RMSE of one fold" if i == 0 else None)
    ax2.scatter(means[name], i, marker="|", s=500, color=TEXT_COLOR, linewidth=2.5, zorder=4,
                label="Mean RMSE" if i == 0 else None)
ax2.set_yticks(positions, order, color=TEXT_COLOR)
ax2.invert_yaxis()
ax2.set_xlim(0, per_fold_table.values.max() * 1.1)
style_axis(ax2)
ax2.set_title("RMSE of each fold (spread = consistency)", loc="left", color=TEXT_COLOR, fontsize=12)
ax2.set_xlabel("RMSE (years)", color=MUTED_TEXT)
ax2.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, labelcolor=MUTED_TEXT)
plt.tight_layout()
plt.show()


## Step 9.4: Observations

The comparison table and charts above show the results from the current run. The best algorithm is the one with the lowest mean RMSE; performance may vary slightly across library versions and runs.


# Step 10: Train the Final Model
## Step 10.1: Select the Best Algorithm

**Cross validation is used to *choose* an algorithm, not to produce the final model.** The 10 models built during cross validation were only used to measure performance and are now thrown away.

We select the algorithm with the **lowest mean RMSE** from the summary table.

In [ ]:
best_algorithm_name = summary.index[0]
best_mean_rmse = summary.loc[best_algorithm_name, "Mean RMSE (years)"]
best_std_rmse = summary.loc[best_algorithm_name, "Std RMSE (years)"]

print(f"Best algorithm : {best_algorithm_name}")
print(f"Expected RMSE  : {best_mean_rmse:.3f} (+/- {best_std_rmse:.3f}) years")


## Step 10.2: Train the Best Algorithm on All the Data

After selecting the best algorithm from cross-validation, train it on the full available Abalone dataset. The cross-validation RMSE is used as an estimate of performance on unseen samples.


In [ ]:
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", algorithms[best_algorithm_name]),
])

final_model.fit(X, y)

print(f"Final {best_algorithm_name} model trained on {len(X):,} abalones.")


## Step 10.3: Save the Trained Model

The saved file contains the complete preprocessing pipeline and regression model, so it can predict directly from raw abalone measurements. In Colab, set `DOWNLOAD_MODEL = True` to download the model file.


In [ ]:
MODEL_FILE = "abalone_age_model_full_dataset.pkl"
DOWNLOAD_MODEL = False

with open(MODEL_FILE, "wb") as f:
    pickle.dump(final_model, f)

print(f"Trained model saved as '{MODEL_FILE}'")

if IN_COLAB and DOWNLOAD_MODEL:
    files.download(MODEL_FILE)


# Step 11: Execute the Application Phase
## Step 11.1: Take Input from User

By default, the notebook uses example measurements so **Run all** works without stopping. To enter your own measurements, set `USE_INTERACTIVE_INPUT = True` and run the cell again.


In [ ]:
USE_INTERACTIVE_INPUT = False

example_abalone = {
    "Length": 0.455,
    "Diameter": 0.365,
    "Height": 0.095,
    "Whole weight": 0.514,
    "Shucked weight": 0.2245,
    "Viscera weight": 0.1010,
    "Shell weight": 0.150,
    "Sex": "M",
}

def get_abalone_input():
    print("Enter Abalone Measurements (ranges seen in the data are shown):")
    for feature in numeric_features:
        print(f"  {feature:<16}: {X[feature].min():.4f} - {X[feature].max():.4f}")
    print("Available Sex values:", sorted(X["Sex"].dropna().unique().tolist()))

    abalone_input = {
        "Length": float(input("Length: ")),
        "Diameter": float(input("Diameter: ")),
        "Height": float(input("Height: ")),
        "Whole weight": float(input("Whole weight: ")),
        "Shucked weight": float(input("Shucked weight: ")),
        "Viscera weight": float(input("Viscera weight: ")),
        "Shell weight": float(input("Shell weight: ")),
        "Sex": input("Sex (M/F/I): ").strip().upper(),
    }
    return abalone_input

user_input = get_abalone_input() if USE_INTERACTIVE_INPUT else example_abalone

print("Abalone measurements:")
for feature, value in user_input.items():
    print(f"  {feature:<17}: {value}")


## Step 11.2: Convert User Input into Feature Vector

The input must contain the same features, in the same order, as the training data. The saved pipeline handles imputation, scaling, and encoding automatically.


In [ ]:
feature_vector = pd.DataFrame([user_input])[numeric_features + categorical_features]
feature_vector


## Step 11.3: Load the Saved Model

In [ ]:
with open(MODEL_FILE, "rb") as f:
    loaded_model = pickle.load(f)

print(f"Model loaded from '{MODEL_FILE}': {loaded_model.named_steps['regressor'].__class__.__name__}")

## Step 11.4: Model Prediction

The trained model predicts the new abalone's estimated age in years and reports the cross-validation RMSE as a rough indication of typical prediction error.


In [ ]:
predicted_age = loaded_model.predict(feature_vector)[0]

print(f"Predicted Abalone Age: {predicted_age:.2f} years")
print(f"Typical error (cross-validation RMSE of {best_algorithm_name}): about {best_mean_rmse:.2f} years")


# Step 12: Execute the Feedback Phase

Potential improvements include tuning the regression algorithms' hyperparameters, collecting more measurements, and evaluating the model on a separate held-out dataset. Remember that the target age is an estimate derived from ring count (`Rings + 1.5`).


# Conclusion

This notebook compares five regression algorithms on the Abalone dataset using 10-fold cross-validation and RMSE. It selects the algorithm with the lowest mean RMSE, trains it on the full dataset, saves the model, and predicts the estimated age in years for a new abalone based on its physical measurements.
